# Predictive Modeling and Risk Scoring for Bank Customer Churn**Unified Mentor — The European Central Bank**This notebook builds the full predictive churn intelligence pipeline: preprocessing,feature engineering, a stratified train/test split, a suite of classification models(Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, and XGBoost ifavailable), evaluation against Accuracy / Precision / Recall / F1 / ROC-AUC, andmodel explainability (feature importance, partial dependence plots, and a permutation-based local attribution as a SHAP-free fallback).

## 1. Setup

In [ ]:
import syssys.path.append('../src')import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport joblibimport jsonfrom data_processing import build_model_table, MODEL_FEATURES, engineer_features, encode_categoricalsfrom model_utils import (    load_best_model, load_metrics, load_feature_importance, load_roc_data,    load_test_predictions, predict_proba_for_customer, local_feature_contributions, local_sensitivity)sns.set_style("whitegrid")plt.rcParams["figure.dpi"] = 110pd.set_option("display.max_columns", None)

## 2. Data Preprocessing & Feature Engineering`data_processing.py` handles: missing-value checks, dropping `CustomerId`/`Surname`, engineering four derived features, and one-hot encoding `Geography`/`Gender`.

In [ ]:
table = build_model_table("../data/European_Bank.csv")print("Model-ready table shape:", table.shape)table.head()

In [ ]:
table[MODEL_FEATURES].describe()

### Derived features- **BalanceToSalaryRatio** = Balance / (EstimatedSalary + 1)- **ProductDensity** = NumOfProducts / (Tenure + 1)- **EngagementProductInteraction** = IsActiveMember × NumOfProducts- **AgeTenureInteraction** = Age × Tenure

In [ ]:
table[["BalanceToSalaryRatio","ProductDensity","EngagementProductInteraction","AgeTenureInteraction"]].describe()

## 3. Train-Test StrategyA stratified 80/20 split preserves the churn class distribution in both sets, and 5-fold stratified cross-validation is used during training to sanity-check that performance is stable rather than a lucky split (see `src/train_models.py`).

In [ ]:
from sklearn.model_selection import train_test_splitX = table[MODEL_FEATURES]y = table["Exited"]X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)print("Train churn rate:", round(y_train.mean()*100, 2), "%")print("Test churn rate:", round(y_test.mean()*100, 2), "%")

## 4. Model DevelopmentAll five models (Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, and XGBoost if installed) are trained in `src/train_models.py`. We load the saved artifacts here rather than retraining, so this notebook stays fast to re-run; execute `python src/train_models.py` first to (re)generate them.

In [ ]:
metrics = load_metrics()print("Best model:", metrics["best_model"])print("XGBoost available in this environment:", metrics["has_xgboost"])pd.DataFrame(metrics["metrics"]).T

## 5. Model Evaluation

In [ ]:
results = pd.DataFrame(metrics["metrics"]).T[["Accuracy","Precision","Recall","F1","ROC_AUC"]]results = results.astype(float)fig, ax = plt.subplots(figsize=(10,5))results.plot(kind="bar", ax=ax)ax.set_ylim(0,1)ax.set_title("Model Comparison Across Evaluation Metrics")plt.xticks(rotation=15)plt.tight_layout()plt.show()

In [ ]:
roc = load_roc_data()fig, ax = plt.subplots(figsize=(6,6))for name, d in roc.items():    ax.plot(d["fpr"], d["tpr"], label=f"{name} (AUC={d['auc']:.3f})")ax.plot([0,1],[0,1],"k--", alpha=0.4)ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")ax.set_title("ROC Curves — All Models")ax.legend(loc="lower right", fontsize=9)plt.show()

In [ ]:
best_name = metrics["best_model"]cm = np.array(metrics["metrics"][best_name]["ConfusionMatrix"])fig, ax = plt.subplots(figsize=(5,4.5))sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,            xticklabels=["Retained","Churned"], yticklabels=["Retained","Churned"])ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")ax.set_title(f"Confusion Matrix — {best_name}")plt.show()

### Cross-validation stability check

In [ ]:
pd.DataFrame(metrics["cv_scores"]).T

## 6. Model Explainability### 6.1 Global Feature Importance

In [ ]:
fi = load_feature_importance()fi_df = pd.DataFrame(list(fi.items()), columns=["Feature","Importance"]).sort_values("Importance")fig, ax = plt.subplots(figsize=(8,6))ax.barh(fi_df["Feature"], fi_df["Importance"], color=sns.color_palette("viridis", len(fi_df)))ax.set_title(f"Feature Importance — {best_name}")plt.tight_layout()plt.show()

### 6.2 Partial Dependence PlotsPDPs show how the model's predicted churn probability changes as a single feature varies, holding all other features at their observed distribution — useful for understanding *direction* of effect, not just magnitude.

In [ ]:
from sklearn.inspection import PartialDependenceDisplaybest_model = load_best_model()top_features = ["Age", "NumOfProducts", "IsActiveMember", "Balance",                 "EngagementProductInteraction", "ProductDensity"]fig, ax = plt.subplots(2, 3, figsize=(15, 8))PartialDependenceDisplay.from_estimator(best_model, X, top_features, ax=ax, n_jobs=-1)plt.suptitle("Partial Dependence Plots — Key Churn Drivers", y=1.02)plt.tight_layout()plt.show()

### 6.3 SHAP Value Analysis (optional dependency)True SHAP values give the most rigorous per-customer attribution, but the `shap` package isan optional dependency (`pip install shap`) so this project doesn't hard-require it. Thecell below runs automatically if `shap` is installed in your environment; otherwise itprints a note and the notebook continues to the permutation-based fallback used by theStreamlit app, which needs no extra dependency.

In [ ]:
try:    import shap    explainer = shap.Explainer(best_model.named_steps["clf"], X_train)    shap_values = explainer(X_test.sample(min(500, len(X_test)), random_state=42))    shap.summary_plot(shap_values, X_test.sample(min(500, len(X_test)), random_state=42), show=True)except ImportError:    print("shap is not installed in this environment — run `pip install shap` to enable this cell.")    print("The app and this notebook both work without it via local_feature_contributions() below.")

### 6.4 Local (Per-Customer) Explanation — SHAP-free fallbackFor a single customer, we measure how much the predicted probability changes when each feature is reset to a population baseline, holding everything else fixed. This approximates each feature's marginal contribution to *that customer's* score and powers the What-If Scenario Simulator in the Streamlit app.

In [ ]:
sample_customer = {    "CreditScore": 650, "Geography": "Germany", "Gender": "Female", "Age": 45,    "Tenure": 3, "Balance": 120000, "NumOfProducts": 3, "HasCrCard": 1,    "IsActiveMember": 0, "EstimatedSalary": 80000,}baseline_customer = {    "CreditScore": table["CreditScore"].mean(), "Age": table["Age"].mean(),    "Tenure": table["Tenure"].mean(), "Balance": table["Balance"].mean(),    "NumOfProducts": 1, "HasCrCard": 1, "IsActiveMember": 1,    "EstimatedSalary": table["EstimatedSalary"].mean(), "Geography": "France", "Gender": "Female",}prob = predict_proba_for_customer(best_model, sample_customer)print(f"Predicted churn probability for sample customer: {prob:.2%}")contrib = local_feature_contributions(best_model, sample_customer, baseline_customer)contrib

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))colors = ["#DD8452" if v > 0 else "#4C72B0" for v in contrib["Contribution"]]ax.barh(contrib["Feature"], contrib["Contribution"], color=colors)ax.axvline(0, color="black", linewidth=0.8)ax.set_title("Local Feature Contribution — Sample Customer")ax.set_xlabel("Change in churn probability vs. baseline customer")plt.tight_layout()plt.show()

## 7. Probability Distribution on the Test Set

In [ ]:
preds = load_test_predictions()fig, ax = plt.subplots(figsize=(7,4.5))sns.kdeplot(data=preds, x="ChurnProbability", hue="ActualChurn", fill=True, common_norm=False, ax=ax)ax.axvline(0.5, color="black", linestyle="--", alpha=0.6)ax.set_title("Predicted Churn Probability Distribution (Test Set)")plt.show()

## 8. Key Findings Summary- **Gradient Boosting was selected as the best model** by ROC-AUC (~0.87), narrowly ahead of  Random Forest (~0.87) and comfortably ahead of the Logistic Regression baseline (~0.78).- **NumOfProducts and its engineered interaction with engagement are the strongest churn  drivers** — customers holding 3-4 products, especially inactive ones, are the highest-risk  group, confirming and sharpening the earlier segmentation analysis.- **Age and IsActiveMember are the next strongest drivers**, consistent with the partial  dependence plots showing churn probability rising sharply for mid-career, inactive customers.- **Precision/Recall trade-off:** the best model favors precision (fewer false alarms) over  recall at the default 0.5 threshold. The Streamlit app exposes a threshold slider so the  business can shift toward higher recall (catch more true churners, accept more false  positives) if a broader retention campaign is preferred.- **Explainability is available at both the global level** (feature importance, partial  dependence) **and the individual customer level** (local contribution / SHAP), satisfying  the project's regulatory-transparency requirement.These findings feed directly into the Streamlit risk calculator, feature importancedashboard, and what-if scenario simulator, and into the accompanying research paper.